# Analyze train/val/test splits
Analyze which timestamps we want to use to split the datasets into train/val/test datasets.

* Train: 70%
* Val: 15%
* Test: 15%

In [15]:
# Setup
import pandas as pd

data_dir = "../../data"

# Load data
tweet = pd.read_parquet(data_dir + "/tweets_2019.parquet")
mention = pd.read_parquet(data_dir + "/mention_rel.parquet")
users = pd.read_parquet(data_dir + "/users_2019.parquet")

## Check data density over time

In [2]:
# Check tweet density
tweet["month"] = tweet["created_at"].dt.to_period("M")
print("Tweet:")
print(tweet["month"].value_counts().sort_index())

# Check mention density
mention["month"] = mention["created_at"].dt.to_period("M")
print("Mention:")
print(mention["month"].value_counts().sort_index())

Tweet:
month
2019-01    45725
2019-02    50118
2019-03    53946
2019-04    54424
2019-05    50312
2019-06    50811
2019-07    49966
2019-08    48698
2019-09    53728
2019-10    56490
2019-11    54006
2019-12    47753
Freq: M, Name: count, dtype: int64
Mention:
month
2019-01    21645
2019-02    24140
2019-03    29017
2019-04    27775
2019-05    27754
2019-06    29494
2019-07    26692
2019-08    24505
2019-09    28696
2019-10    32835
2019-11    32009
2019-12    25358
Freq: M, Name: count, dtype: int64


# 1. Data split for mention link prediction

## Use quantile on tweet creation times to split the dataset

In [3]:
# Use quantile on tweet creation times
tweet_times = tweet["created_at"].sort_values()
val_timestamp = tweet_times.quantile(0.70)
test_timestamp = tweet_times.quantile(0.85)

print(f"val timestamp: {val_timestamp}")
print(f"test timestamp: {test_timestamp}")

val timestamp: 2019-09-16 22:14:49.400000
test timestamp: 2019-11-06 16:25:56


## Check the number of mention edges in each split

In [4]:
total_mentions = len(mention)
val_timestamp = '2019-09-01'
test_timestamp = '2019-11-01'

for name, m_mask in [
    (
        "train",
        mention["created_at"] < val_timestamp,
    ),
    (
        "val",
        (mention["created_at"] >= val_timestamp) & (mention["created_at"] < test_timestamp),
    ),
    (
        "test",
        mention["created_at"] >= test_timestamp,
    ),
]:
    m_pct = m_mask.sum() / total_mentions * 100
    print(f"{name}: mentions {m_pct:.1f}%")

train: mentions 64.0%
val: mentions 18.7%
test: mentions 17.4%
